In [1]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, avg, count, when, sum
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from snowflake.ml.data.data_connector import DataConnector

from snowflake.snowpark import Session
from credentials import params

from snowflake.snowpark.types import (
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
    StringType
)

from snowflake.ml.registry import Registry

In [2]:
session = Session.builder.configs(params).create()
session.use_database("HOUSING_PRICE_PROJECT")
session.use_schema("ML_LAYER")

In [3]:
#df_sp = session.table("HOUSING_PRICE_PROJECT.STAGING_LAYER.RAW_DATA")
#df = df_sp.to_pandas()

X_train_sp = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TRAIN_SET")
#X_train_pd = X_train_sp.to_pandas()

#X_test_sp = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TEST_SET")
#X_test = X_test_sp.to_pandas()

In [5]:
X_train_sp.columns

['CITY',
 'LOCALITY',
 'LOCALITY_TIER',
 'PROPERTY_TYPE',
 'BALCONIES',
 'CARPET_AREA',
 'FLOOR_NUMBER',
 'TOTAL_FLOORS',
 'FLOOR_CATEGORY',
 'FACING',
 'FURNISHING_STATUS',
 'PROPERTY_AGE',
 'PARKING_SPACES',
 'SECURITY_SCORE',
 'GYM_AVAILABLE',
 'SWIMMING_POOL',
 'POWER_BACKUP',
 'LIFT_AVAILABLE',
 'MAINTENANCE_FEE_MONTHLY',
 'DISTANCE_TO_CITY_CENTER_KM',
 'DISTANCE_TO_METRO_KM',
 'NEARBY_SCHOOLS',
 'NEARBY_HOSPITALS',
 'TRANSACTION_TYPE',
 'PRICE_IN_LAKHS']

In [6]:
#res = X_train_pd

#print(type(res), end = "\n\n")
#res_cols = list(res.columns)
#print(res_cols, end = "\n\n")
#print(res.info(), end = "\n\n")

#for i in res_cols:
#    print(i)
#    print(res.loc[0,i], end = "\n\n")

In [7]:
#X_train = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TRAIN_SET")
#X_train = X_train.drop('"price_in_lakhs"', '"property_id"', '"price_category"', 'SOURCE_FILE')

#for col in X_train.columns:
#    if col[0] =='"':
#        X_train = X_train.with_column_renamed(col, col[1:-1])

In [8]:
#X_train.columns

In [9]:
"""
registry = Registry(
    session=session,
    database_name="HOUSING_PRICE_PROJECT",
    schema_name="ML_LAYER"
)

model_version = registry \
    .get_model("PREPROCESSING_PIPELINE") \
    .version("LAST")

predictions = model_version.run(
    X_train, #X_train_long
    function_name="transform"
)
"""

'\nregistry = Registry(\n    session=session,\n    database_name="HOUSING_PRICE_PROJECT",\n    schema_name="ML_LAYER"\n)\n\nmodel_version = registry     .get_model("PREPROCESSING_PIPELINE")     .version("LAST")\n\npredictions = model_version.run(\n    X_train, #X_train_long\n    function_name="transform"\n)\n'

In [10]:
#predictions.columns

In [11]:
#predictions.select('"""CITY_Bangalore"""').show()

In [12]:
#for col in predictions.columns:
    #print(col[3:-3].replace(" ", "_").replace("-", "_"))
#    if col[0] =='"':
#        predictions = predictions.with_column_renamed(col, col.replace('"', '')) #.replace(" ", "_").replace("-", "_"))

#predictions.columns

# ML Training

In [4]:
from snowflake.ml.jobs import remote

In [5]:
from snowflake.ml.jobs import remote
session.use_database("HOUSING_PRICE_PROJECT")
session.use_schema("ML_LAYER")

@remote(
    "ml_job_computerpool",
    stage_name="HOUSING_PRICE_PROJECT.ML_LAYER.MLJOBS_STAGE",
    session=session,
    target_instances=4
)
def training_and_eval(train_name, test_name):

    from datetime import datetime
    from snowflake.ml.data.data_connector import DataConnector
    from snowflake.ml.modeling.distributors.xgboost.xgboost_estimator import XGBScalingConfig
    from snowflake.ml.runtime_cluster import scale_cluster
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    from snowflake.ml.modeling.distributors.xgboost.xgboost_estimator import XGBEstimator    
    from snowflake.snowpark.types import StringType
    import time
    print("\nlibrerie importate ****************************************************")
    
    # SESSION, DATASET AND FEATURE LIST ************************************************************************
    job_session = get_active_session()
    
    
    # train tab ****************************************
    train_tab = job_session.table(train_name)
    #train_tab = train_tab.drop('"property_id"', '"price_category"', 'SOURCE_FILE') # <--- REMOVED ON 29/08
    
    #for col in train_tab.columns: # <--- REMOVED ON 29/08
    #    if col[0] =='"':
    #        train_tab = train_tab.with_column_renamed(col, col[1:-1])
    print("\ntrain tab importata ****************************************************")
    
    # test tab ****************************************
    test_tab = job_session.table(test_name)
    #test_tab = test_tab.drop('"property_id"', '"price_category"', 'SOURCE_FILE') # <--- REMOVED ON 29/08
    
    #for col in test_tab.columns: # <--- REMOVED ON 29/08
    #    if col[0] =='"':
    #        test_tab = test_tab.with_column_renamed(col, col[1:-1])
    
    print("\ntest tab importata ****************************************************")

    # target col ****************************************
    target_col_name = 'PRICE_IN_LAKHS'
    
    # PREPROCESSING using the pipeline from model registry *******************************************************
    registry = Registry(
        session=job_session,
        database_name="HOUSING_PRICE_PROJECT",
        schema_name="ML_LAYER"
    )
    model_version = registry.get_model("PREPROCESSING_PIPELINE").version("LAST")    
    
    # train tab ****************************************
    train_tab_prepr = model_version.run(
        train_tab,
        function_name="transform"
    )
    print("\ntrain tab preprocessato ****************************************************")
    # test tab ****************************************
    test_tab_prepr = model_version.run(
        test_tab,
        function_name="transform"
    ) 
    print("\ntest tab preprocessato ****************************************************")
    # ADJUSTING THE PREPROCESSED DATASET *************************************************************************
    
    
    import re
    def sanitize_column_name(column):
        column = column.replace('+', 'plus')
        column = re.sub(r'[()"]', '', column)
        column = re.sub(r'[\s-]', '_', column)                   
        column = re.sub(r'[^a-zA-Z0-9_]', '', column)
        return column

    # train tab ****************************************
    for column in train_tab_prepr.columns:
        train_tab_prepr = train_tab_prepr.with_column_renamed(column, sanitize_column_name(column))
    

    # test tab ****************************************
    for column in test_tab_prepr.columns:
        test_tab_prepr = test_tab_prepr.with_column_renamed(column, sanitize_column_name(column))
    
    
    cat_cols = [
        field.name
        for field in train_tab_prepr.schema.fields
        if isinstance(field.datatype, StringType)    
    ]
    
    train_tab_prepr = train_tab_prepr.drop(cat_cols)
    test_tab_prepr = test_tab_prepr.drop(cat_cols)

    print("\nprocessed tabs aggiustate ****************************************************")

    #CREATING FEATURE LIST **************************************************************************************
    features = train_tab_prepr.columns
    features.remove(target_col_name)

    print(train_tab_prepr.columns)
    print(train_tab_prepr.to_pandas().head(5))
    return
    
    train_connector = DataConnector.from_dataframe(train_tab_prepr)

    # RETRIEVE BEST HYPERPARAM **************************************************************************************
    # retrieve the latest tuning run id
    tuning_run_id = job_session.sql("""
        SELECT TUNING_RUN_ID
        FROM XGBOOST_BEST_PARAMETERS
        ORDER BY TIMESTAMP DESC
        LIMIT 1
    """).to_pandas()["TUNING_RUN_ID"][0]
    
    # use this line in case you want to use hyperparam from a specific tuning run
    # tuning_run_id = ""

    # retrieve the best hyperparam using "tuning_run_id"
    params_df = job_session.sql(f"""
        SELECT MAX_DEPTH, LEARNING_RATE, N_ESTIMATORS, MIN_CHILD_WEIGHT, SUBSAMPLE
        FROM XGBOOST_BEST_PARAMETERS
        WHERE TUNING_RUN_ID = '{tuning_run_id}';
    """).to_pandas()



    print("\nabout to start with XGBEstim ****************************************************")
    
    estimator = XGBEstimator(
        params={
            "max_depth": int(params_df["MAX_DEPTH"][0]),
            "learning_rate": params_df["LEARNING_RATE"][0],
            "min_child_weight": int(params_df["MIN_CHILD_WEIGHT"][0]),
            "subsample": int(params_df["SUBSAMPLE"][0])
        },
        n_estimators = int(params_df["N_ESTIMATORS"][0]),
        
        objective = "reg:squarederror",
        scaling_config = XGBScalingConfig(
            num_workers = 1
        )
    )
    print("\nML dichiarato ****************************************************")

    train_start = time.time()
    estimator.fit(
        dataset=train_connector,
        input_cols=features,
        label_col=target_col_name
    )
    train_end = time.time()
    print("\nML addestrato ****************************************************")
    


    
    # train tab ****************************************
    # Use a train subset for metrics to avoid materializing the full dataset in Pandas
    SIZE_TRAIN_SAMP = 15_000 
    
    X_train_metric_samp_pd = train_tab_prepr.sample(n=SIZE_TRAIN_SAMP).to_pandas()
    target_train_metric_samp = X_train_metric_samp_pd[target_col_name]
    X_train_metric_samp_pd = X_train_metric_samp_pd.drop([target_col_name], axis = 1)
    

    # test tab ****************************************
    X_test_pd = test_tab_prepr.to_pandas()
    target_test = X_test_pd[target_col_name]
    X_test_pd = X_test_pd.drop([target_col_name], axis = 1) 
    print("\npandas dataframes creati ****************************************************")
    
            
    # train tab ****************************************            
    predictions_train_metric_samp = estimator.predict(X_train_metric_samp_pd)
       
    rmse_train = mean_squared_error(
        y_true = target_train_metric_samp,
        y_pred = predictions_train_metric_samp
    )

    mae_train = mean_absolute_error(
        y_true = target_train_metric_samp,
        y_pred = predictions_train_metric_samp
    )

    r2_train = r2_score(
        y_true = target_train_metric_samp,
        y_pred = predictions_train_metric_samp
    )
    
    print("\ntrain prediction e rmse fatti ****************************************************")   

    # test tab ****************************************    
    pred_test_start = time.time()
    predictions_test = estimator.predict(X_test_pd)
    pred_test_end = time.time()
       
    rmse_test = mean_squared_error(
        y_true = target_test,
        y_pred = predictions_test
    )

    mae_test = mean_absolute_error(
        y_true = target_test,
        y_pred = predictions_test
    )

    r2_test = r2_score(
        y_true = target_test,
        y_pred = predictions_test
    )
    print("\ntest prediction e rmse fatti ****************************************************")
    
    # RECORDING TIMESTAMP AT THE END OF THE PROCESS
    TIMESTAMP = datetime.now()
   
    # INSERT MODEL INTO THE SNOWFLAKE REGISTRY
    
    model_version = registry.log_model(
        model = estimator.get_booster(),
        model_name = "HOUSING_PRICE_XGBOOST",
        version_name = f"run_{TIMESTAMP.strftime('%Y%m%d_%H%M%S')}",
        #comment = "First version of the housing price model",
        metrics = {
            'RMSE_TRAIN' : rmse_train,
            'RMSE_TEST' : rmse_test,
            'MAE_TRAIN' : mae_train,
            'MAE_TEST' : mae_test,
            'R2_TRAIN' : r2_train,
            'R2_TEST' : r2_test,
            'TRAIN_RUN_ID' : TIMESTAMP.strftime("%Y%m%d_%H%M%S")
        },
        sample_input_data = X_test_pd.head(),
        target_platforms=[
            "WAREHOUSE",
            "SNOWPARK_CONTAINER_SERVICES"
        ]
    )  


    # INSERT DATA INTO XGBOOST_TRAINING_RESULTS TAB ******************************

    # results gathering
    results_dict = {
        'TRAIN_RUN_ID' : TIMESTAMP.strftime("%Y%m%d_%H%M%S"),
        'TIMESTAMP' : TIMESTAMP,
        'HYPERPARAMETER_SET_ID' : tuning_run_id,
        'RMSE_TRAIN' : rmse_train,
        'RMSE_TEST' : rmse_test,
        'MAE_TRAIN' : mae_train,
        'MAE_TEST' : mae_test,
        'R2_TRAIN' : r2_train,
        'R2_TEST' : r2_test,
        'TRAIN_SET_SIZE' : train_tab.count(),
        'TRAIN_METRIC_SAMPLE_SIZE' : SIZE_TRAIN_SAMP,
        'TEST_SET_SIZE' : test_tab.count(),
        'TARGET_COLUMN' : target_col_name,
        'N_FEATURES_BEFORE_ENCODING' : len(train_tab.columns), 
        'N_FEATURES_AFTER_ENCODING' : len(train_tab_prepr.columns),
        'TRAINING_TIME_S' : train_end - train_start,
        'PREDICTION_TIME_TESTSET_S' : pred_test_end - pred_test_start
    }


    # query creation
    cols_query = ""
    values_query = ""
    for key in results_dict:
        cols_query = cols_query + key + ", "
        if type(results_dict[key]) == int or type(results_dict[key]) == float:
            values_query = values_query + str(results_dict[key]) + ", "
        else:
            values_query = values_query + "'" + str(results_dict[key]) + "', "
    
    
    query = f"""INSERT INTO XGBOOST_TRAINING_RESULTS ({cols_query[:-2]})
    VALUES ({values_query[:-2]})"""
    
    # insertion
    job_session.sql(query).collect()
     

In [6]:
import time
start = time.time()

job = training_and_eval("HOUSING_PRICE_PROJECT.ML_LAYER.TRAIN_SET", "HOUSING_PRICE_PROJECT.ML_LAYER.TEST_SET") 
job.wait()

end = time.time()

print(job.status)
print(f"{(end - start) / 60 :.2f} min")

DONE
5.38 min


In [7]:
job.show_logs(verbose=True)

2026-09-01T18:33:33.876Z	info	otelconftelemetry/tracer.go:47	Internal trace telemetry disabled	{"resource": {"service.instance.id": "73dbe650-da47-4692-8606-461712eac356", "service.name": "otelcol-contrib", "service.version": "0.156.0"}}
2026-09-01T18:33:33.876Z	warn	builders/builders.go:40	"otlp" alias is deprecated; use "otlp_grpc" instead	{"resource": {"service.instance.id": "73dbe650-da47-4692-8606-461712eac356", "service.name": "otelcol-contrib", "service.version": "0.156.0"}, "otelcol.component.id": "otlp", "otelcol.component.kind": "exporter", "otelcol.signal": "metrics"}
2026-09-01T18:33:33.897Z	info	service@v0.156.0/service.go:256	Starting otelcol-contrib...	{"resource": {"service.instance.id": "73dbe650-da47-4692-8606-461712eac356", "service.name": "otelcol-contrib", "service.version": "0.156.0"}, "Version": "0.156.0", "NumCPU": 8}
2026-09-01T18:33:33.897Z	info	extensions/extensions.go:41	Starting extensions...	{"resource": {"service.instance.id": "73dbe650-da47-4692-8606-461

In [8]:
job.status

'DONE'

In [20]:
type(job.status)

str

In [ ]:
#************************************

In [ ]:
cols_before

In [ ]:
import re
def sanitize_column_name(column):
    column = column.replace('+', 'plus')
    column = re.sub(r'[()"]', '', column)
    column = re.sub(r'[\s-]', '_', column)                   
    column = re.sub(r'[^a-zA-Z0-9_]', '', column)
    return column

cols_after = []
for i in cols_before:
    cols_after.append(sanitize_column_name(i))
cols_after

In [ ]:
train_tab_prepr.columns

# Creazione Tabella per Store risultati

In [24]:
session.sql("""
    CREATE OR REPLACE TABLE XGBOOST_TRAINING_RESULTS (

        train_run_id VARCHAR,
        timestamp TIMESTAMP_NTZ,
        hyperparameter_set_id VARCHAR,
        
        rmse_train DOUBLE,
        rmse_test DOUBLE,

        mae_train DOUBLE,
        mae_test DOUBLE,
        r2_train DOUBLE,
        r2_test DOUBLE,

        train_set_size BIGINT,
        train_metric_sample_size BIGINT,
        test_set_size BIGINT,
        
        target_column VARCHAR,
        n_features_before_encoding BIGINT,
        n_features_after_encoding BIGINT,
        
        training_time_s DOUBLE,
        prediction_time_testset_s DOUBLE
    );
""").collect()

[Row(status='Table XGBOOST_TRAINING_RESULTS successfully created.')]

In [ ]:
session.sql("""
    SELECT * FROM XGBOOST_TRAINING_RESULTS
""").to_pandas()

In [ ]:
a = "train_run_id, timestamp, hyperparameter_set_id, rmse_train, rmse_test, mae_train, mae_test, r2_train, r2_test, train_set_size, train_metric_sample_size, test_set_size, target_column, n_features_before_encoding, n_features_after_encoding, training_time_s, prediction_time_s"
a.upper().split()

# Altro

In [ ]:
from datetime import datetime
#from snowflake.ml.data.data_connector import DataConnector
#from snowflake.ml.modeling.distributors.xgboost.xgboost_estimator import XGBScalingConfig
#from snowflake.ml.runtime_cluster import scale_cluster
#from snowflake.ml.modeling.tune import get_tuner_context
#from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
#from snowflake.ml.modeling.distributors.xgboost.xgboost_estimator import XGBEstimator    
from snowflake.snowpark.types import StringType
import time
print("\nlibrerie importate ****************************************************")


# train tab ****************************************
train_tab = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TRAIN_SET")
train_tab = train_tab.drop('"property_id"', '"price_category"', 'SOURCE_FILE')

for col in train_tab.columns:
    if col[0] =='"':
        train_tab = train_tab.with_column_renamed(col, col[1:-1])
print("\ntrain tab importata ****************************************************")

# test tab ****************************************
test_tab = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TEST_SET")
test_tab = test_tab.drop('"property_id"', '"price_category"', 'SOURCE_FILE')

for col in test_tab.columns:
    if col[0] =='"':
        test_tab = test_tab.with_column_renamed(col, col[1:-1])

print("\ntest tab importata ****************************************************")

# target col ****************************************
target_col_name = 'PRICE_IN_LAKHS'

# PREPROCESSING using the pipeline from model registry *******************************************************
registry = Registry(
    session=session,
    database_name="HOUSING_PRICE_PROJECT",
    schema_name="ML_LAYER"
)
model_version = registry.get_model("PREPROCESSING_PIPELINE").version("LAST")    

# train tab ****************************************

train_tab_prepr = model_version.run(
    train_tab,
    function_name="transform"
)


print("\ntrain tab preprocessato ****************************************************")




# ADJUSTING THE PREPROCESSED DATASET *************************************************************************
# train tab ****************************************

cols_before = train_tab_prepr.columns

print(f"\ncolonne pre-sistemazione:\n{train_tab_prepr.columns}\n")


import re
def sanitize_column_name(column):
    column = column.replace('+', 'plus')
    column = re.sub(r'[()"]', '', column)
    column = re.sub(r'[\s-]', '_', column)                   
    column = re.sub(r'[^a-zA-Z0-9_]', '', column)
    return column

for column in train_tab_prepr.columns:
    train_tab_prepr = train_tab_prepr.with_column_renamed(column, sanitize_column_name(column))


print(f"\ncolonne post-sistemazione:\n{train_tab_prepr.columns}\n")



cat_cols = [
    field.name
    for field in train_tab_prepr.schema.fields
    if isinstance(field.datatype, StringType)    
]


train_tab_prepr = train_tab_prepr.drop(cat_cols)

print("\nprocessed tabs aggiustate ****************************************************")

#CREATING FEATURE LIST **************************************************************************************
features = train_tab_prepr.columns
features.remove(target_col_name)


In [ ]:
params_df = session.sql("""
    SELECT MAX_DEPTH, LEARNING_RATE, N_ESTIMATORS, MIN_CHILD_WEIGHT, SUBSAMPLE, TIMESTAMP
    FROM XGBOOST_BEST_PARAMETERS
    ORDER BY TIMESTAMP DESC
    LIMIT 1
""").to_pandas()

In [ ]:
params_df["N_ESTIMATORS"][0]

In [ ]:
params_df

In [ ]:
session.sql("""SELECT * FROM XGBOOST_BEST_PARAMETERS""").to_pandas()

In [ ]:
tuning_run_id = session.sql("""
    SELECT TUNING_RUN_ID
    FROM XGBOOST_BEST_PARAMETERS
    ORDER BY TIMESTAMP DESC
    LIMIT 1
""").to_pandas()["TUNING_RUN_ID"][0]


# USE THIS LINE IN CASE YOU WANT TO USE HYPERPARAM FROM A SPECIFIC TUNING RUN
#tuning_run_id = ""


params_df = session.sql(f"""
    SELECT MAX_DEPTH, LEARNING_RATE, N_ESTIMATORS, MIN_CHILD_WEIGHT, SUBSAMPLE
    FROM XGBOOST_BEST_PARAMETERS
    WHERE TUNING_RUN_ID = '{tuning_run_id}';
""").to_pandas()

In [ ]:
params_df

In [ ]:
from datetime import datetime
results_dict = {
    'TRAIN_RUN_ID' : datetime.now().strftime("%Y%m%d_%H%M%S"),
    'TIMESTAMP' : datetime.now(),
    'HYPERPARAMETER_SET_ID' : "2012312131",
    'RMSE_TRAIN' : 1.1,
    'RMSE_TEST' : 2.2,
    'MAE_TRAIN' : 3.3,
    'MAE_TEST' : 4.4,
    'R2_TRAIN' : 5.5,
    'R2_TEST' : 6.6,
    'TRAIN_SET_SIZE' : 7,
    'TRAIN_METRIC_SAMPLE_SIZE' : 8,
    'TEST_SET_SIZE' : 9,
    'TARGET_COLUMN' : "target_col_name",
    'N_FEATURES_BEFORE_ENCODING' : 10, 
    'N_FEATURES_AFTER_ENCODING' : 11,
    'TRAINING_TIME_S' : 12.3,
    'PREDICTION_TIME_TESTSET_S' : 13.1
}

cols_query = ""
values_query = ""
for key in results_dict:
    cols_query = cols_query + key + ", "
    if type(results_dict[key]) == int or type(results_dict[key]) == float:
        values_query = values_query + str(results_dict[key]) + ", "
    else:
        values_query = values_query + "'" + str(results_dict[key]) + "', "


query = f"""INSERT INTO TEMP_TAB ({cols_query[:-2]})
VALUES ({values_query[:-2]})"""

session.sql(query).collect()

In [ ]:
session.sql("SELECT * FROM TEMP_TAB").to_pandas()

In [ ]:
session.sql("""
    CREATE OR REPLACE TEMPORARY TABLE TEMP_TAB (

        train_run_id VARCHAR,
        timestamp TIMESTAMP_NTZ,
        hyperparameter_set_id VARCHAR,
        
        rmse_train DOUBLE,
        rmse_test DOUBLE,

        mae_train DOUBLE,
        mae_test DOUBLE,
        r2_train DOUBLE,
        r2_test DOUBLE,

        train_set_size BIGINT,
        train_metric_sample_size BIGINT,
        test_set_size BIGINT,
        
        target_column VARCHAR,
        n_features_before_encoding BIGINT,
        n_features_after_encoding BIGINT,
        
        training_time_s DOUBLE,
        prediction_time_testset_s DOUBLE
    );
""").collect()

In [ ]:
session.sql("SHOW TABLES").to_pandas()

# Model retrieve and use

In [ ]:
# retrieving models from registry
registry = Registry(
    session=session,
    database_name="HOUSING_PRICE_PROJECT",
    schema_name="ML_LAYER"
)

prepr = registry.get_model("PREPROCESSING_PIPELINE").version("LAST")
xgboost = registry.get_model("HOUSING_PRICE_XGBOOST").version("LAST")
registry.get_model("HOUSING_PRICE_XGBOOST").show_versions()

In [ ]:
# importing tab as snowpark tables
test_tab_sp = session.table("HOUSING_PRICE_PROJECT.ML_LAYER.TEST_SET")
#test_tab_sp = test_tab_sp.drop('"property_id"', '"price_category"', 'SOURCE_FILE') # <--- REMOVED ON 29/08

In [ ]:
# adjusting cols name to match preprocessor requirements
#for col in test_tab_sp.columns: # <--- REMOVED ON 29/08
#    if col[0] =='"':
#        test_tab_sp = test_tab_sp.with_column_renamed(col, col[1:-1])

In [ ]:
# preprocessing data
X_test_prepr = prepr.run(
    test_tab_sp,
    function_name="transform"
)

In [ ]:
# adjusting preprocessed data cols names
import re
from snowflake.snowpark.types import StringType
def sanitize_column_name(column):
    column = column.replace('+', 'plus')
    column = re.sub(r'[()"]', '', column)
    column = re.sub(r'[\s-]', '_', column)                   
    column = re.sub(r'[^a-zA-Z0-9_]', '', column)
    return column

# train tab ****************************************
for column in X_test_prepr.columns:
    X_test_prepr = X_test_prepr.with_column_renamed(column, sanitize_column_name(column))

cat_cols = [
    field.name
    for field in X_test_prepr.schema.fields
    if isinstance(field.datatype, StringType)    
]

X_test_prepr = X_test_prepr.drop(cat_cols)

In [ ]:
# creating dataframe for prediction
X_test_prepr = X_test_prepr
X_test_prepr_df = X_test_prepr.to_pandas()

X_test = X_test_prepr_df.drop('PRICE_IN_LAKHS', axis = 1)
target = X_test_prepr_df['PRICE_IN_LAKHS']

In [ ]:
# predict
predictions = xgboost.run(
    X_test,
    function_name="predict"
)

In [ ]:
a = predictions.select(col('"output_feature_0"')).to_pandas()

In [ ]:
# r2 calculation
from sklearn.metrics import r2_score

r2 = r2_score(
    y_true = target,
    y_pred = predictions
)

print(r2)

In [25]:
session.sql("SELECT * FROM XGBOOST_TRAINING_RESULTS").to_pandas()

,TRAIN_RUN_ID,TIMESTAMP,HYPERPARAMETER_SET_ID,RMSE_TRAIN,RMSE_TEST,MAE_TRAIN,MAE_TEST,R2_TRAIN,R2_TEST,TRAIN_SET_SIZE,TRAIN_METRIC_SAMPLE_SIZE,TEST_SET_SIZE,TARGET_COLUMN,N_FEATURES_BEFORE_ENCODING,N_FEATURES_AFTER_ENCODING,TRAINING_TIME_S,PREDICTION_TIME_TESTSET_S
